# 📦 Notebook 1 — Data Loading & Preparation

**Goal:** Load the raw petrol pump Excel data, understand its structure,
clean it, fix data types, and save a clean version for all future notebooks.

**Steps in this notebook:**
1. Load the Excel file
2. Inspect shape, columns, types
3. Check for missing values
4. Fix data types (dates, integers)
5. Validate business rules (stock logic)
6. Save clean CSV for downstream notebooks

In [2]:
# ── Install required libraries (run once) ──────────────────────
# !pip install pandas openpyxl matplotlib seaborn scikit-learn imbalanced-learn xgboost

In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

Libraries loaded ✅


## Step 1 — Load the Excel file

In [4]:
# ── Load data ──────────────────────────────────────────────────
# Update path to your actual file location
FILE_PATH = '../data/petrol_pump_data_2024_2026.xlsx'

df = pd.read_excel(FILE_PATH)

print(f'Rows    : {df.shape[0]}')
print(f'Columns : {df.shape[1]}')
print(f'\nColumn names:')
for c in df.columns:
    print(f'  {c}')

Rows    : 806
Columns : 14

Column names:
  Date
  Day
  Opening_Stock
  MS_Sold
  HSD1_Sold
  HSD2_Sold
  HSD3_Sold
  Total_Sold
  Closing_Stock
  Cash
  Online
  Card
  Dip
  Refill_Required


## Step 2 — Inspect structure

In [5]:
# First 5 rows
df.head()

,Date,Day,Opening_Stock,MS_Sold,HSD1_Sold,HSD2_Sold,HSD3_Sold,Total_Sold,Closing_Stock,Cash,Online,Card,Dip,Refill_Required
0,01-01-2024,Monday,12000.0,564.0,1684.0,1219.0,899.0,4366.0,7634,155590,151727.0,90617,63,No
1,02-01-2024,Tuesday,7634.0,603.0,1740.0,1230.0,778.0,4351.0,3283,160056,131743.0,87654,27,No
2,03-01-2024,Wednesday,3283.0,441.0,1469.0,1174.0,867.0,3951.0,0,161531,127157.0,73959,1,Yes
3,04-01-2024,Thursday,12000.0,492.0,1632.0,1508.0,981.0,4613.0,7387,219210,127815.0,103870,61,No
4,05-01-2024,Friday,7387.0,485.0,1939.0,1368.0,1002.0,4794.0,2593,196707,153101.0,101182,21,No


In [6]:
# Data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 806 entries, 0 to 805
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Date             806 non-null    object 
 1   Day              806 non-null    object 
 2   Opening_Stock    804 non-null    float64
 3   MS_Sold          803 non-null    float64
 4   HSD1_Sold        802 non-null    float64
 5   HSD2_Sold        804 non-null    float64
 6   HSD3_Sold        799 non-null    float64
 7   Total_Sold       801 non-null    float64
 8   Closing_Stock    806 non-null    int64  
 9   Cash             806 non-null    int64  
 10  Online           805 non-null    float64
 11  Card             806 non-null    int64  
 12  Dip              806 non-null    int64  
 13  Refill_Required  806 non-null    object 
dtypes: float64(7), int64(4), object(3)
memory usage: 88.3+ KB


In [7]:
# Basic statistics for all numeric columns
df.describe().round(2)

,Opening_Stock,MS_Sold,HSD1_Sold,HSD2_Sold,HSD3_Sold,Total_Sold,Closing_Stock,Cash,Online,Card,Dip
count,804.00,803.00,802.00,804.00,799.00,801.00,806.00,806.00,805.00,806.00,806.00
mean,8012.88,567.94,1803.57,1428.70,953.65,4756.11,3644.42,196524.57,149670.38,101815.57,30.26
std,3530.81,102.70,272.13,224.34,184.64,691.23,3045.78,31801.62,25286.24,18633.61,24.94
min,2020.00,315.00,1155.00,789.00,451.00,2835.00,0.00,114051.00,83302.00,53570.00,1.00
25%,5496.00,497.00,1601.25,1267.75,818.00,4278.00,90.50,175022.00,131743.00,88476.50,1.00
50%,7543.00,557.00,1797.00,1414.50,934.00,4698.00,3055.00,194132.50,147499.00,100195.00,25.00
75%,12000.00,634.00,1962.00,1579.25,1086.00,5177.00,6917.00,217883.25,164930.00,113571.00,57.00
max,12000.00,913.00,2714.00,2147.00,1564.00,6869.00,9165.00,303203.00,234398.00,163590.00,76.00


## Step 3 — Check for missing values

In [8]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})

print(missing_df)
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

if df.isnull().sum().sum() == 0:
    print('✅ No missing values found!')
else:
    print('⚠️  Missing values found — filling with forward fill')
    df.fillna(method='ffill', inplace=True)

                 Missing Count  Missing %
Date                         0       0.00
Day                          0       0.00
Opening_Stock                2       0.25
MS_Sold                      3       0.37
HSD1_Sold                    4       0.50
HSD2_Sold                    2       0.25
HSD3_Sold                    7       0.87
Total_Sold                   5       0.62
Closing_Stock                0       0.00
Cash                         0       0.00
Online                       1       0.12
Card                         0       0.00
Dip                          0       0.00
Refill_Required              0       0.00

Total missing values: 24
⚠️  Missing values found — filling with forward fill


## Step 4 — Fix data types

In [9]:
# Convert Date column to proper datetime
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

# Extract useful date features
df['Year']       = df['Date'].dt.year
df['Month']      = df['Date'].dt.month
df['DayOfWeek']  = df['Date'].dt.dayofweek          # 0=Monday, 6=Sunday
df['DayOfYear']  = df['Date'].dt.dayofyear
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
df['Quarter']    = df['Date'].dt.quarter
df['Is_Weekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)

# Encode Day name as number (already have DayOfWeek but keeping Day as string too)
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
df['Day_Num'] = df['Day'].map({d: i for i, d in enumerate(day_order)})

# Encode target: Refill_Required → 0/1
df['Target'] = (df['Refill_Required'] == 'Yes').astype(int)

print('Date features added:')
print(df[['Date','Year','Month','DayOfWeek','Is_Weekend','Quarter','Target']].head(10))

Date features added:
        Date  Year  Month  DayOfWeek  Is_Weekend  Quarter  Target
0 2024-01-01  2024      1          0           0        1       0
1 2024-01-02  2024      1          1           0        1       0
2 2024-01-03  2024      1          2           0        1       1
3 2024-01-04  2024      1          3           0        1       0
4 2024-01-05  2024      1          4           0        1       0
5 2024-01-06  2024      1          5           1        1       1
6 2024-01-07  2024      1          6           1        1       0
7 2024-01-08  2024      1          0           0        1       0
8 2024-01-09  2024      1          1           0        1       1
9 2024-01-10  2024      1          2           0        1       0


## Step 5 — Validate business rules

In [10]:
print('=== Business Rule Validation ===')

# Rule 1: Total_Sold = MS + HSD1 + HSD2 + HSD3
df['_calc_total'] = df['MS_Sold'] + df['HSD1_Sold'] + df['HSD2_Sold'] + df['HSD3_Sold']
total_mismatch = (abs(df['Total_Sold'] - df['_calc_total']) > 10).sum()
print(f'Total_Sold mismatches (>10L diff) : {total_mismatch}')

# Rule 2: No negative closing stock
neg_closing = (df['Closing_Stock'] < 0).sum()
print(f'Negative Closing_Stock rows       : {neg_closing}')

# Rule 3: Opening never exceeds 12000
over_capacity = (df['Opening_Stock'] > 12000).sum()
print(f'Opening_Stock > 12000 rows        : {over_capacity}')

# Rule 4: Refill logic correct
refill_mismatch = ((df['Closing_Stock'] < 2000) != (df['Target'] == 1)).sum()
print(f'Refill label mismatches           : {refill_mismatch}')

# Drop temp column
df.drop(columns=['_calc_total'], inplace=True)

print('\n✅ Validation complete')

=== Business Rule Validation ===
Total_Sold mismatches (>10L diff) : 21
Negative Closing_Stock rows       : 0
Opening_Stock > 12000 rows        : 0
Refill label mismatches           : 0

✅ Validation complete


## Step 6 — Add derived features useful for ML

In [10]:
# Stock ratio — how full is the tank as a percentage
df['Stock_Ratio']        = (df['Closing_Stock'] / 12000).round(4)

# Rolling 7-day average of sales (captures weekly trend)
df['Rolling_7d_Sales']   = df['Total_Sold'].rolling(window=7, min_periods=1).mean().round(0)

# Rolling 3-day average of sales
df['Rolling_3d_Sales']   = df['Total_Sold'].rolling(window=3, min_periods=1).mean().round(0)

# Previous day closing stock (lag feature)
df['Prev_Closing']       = df['Closing_Stock'].shift(1).fillna(12000)

# Previous day total sold
df['Prev_Total_Sold']    = df['Total_Sold'].shift(1).fillna(df['Total_Sold'].mean())

# Days since last refill
refill_mask = df['Target'] == 1
df['Days_Since_Refill'] = refill_mask.cumsum()
df['Days_Since_Refill'] = df.groupby('Days_Since_Refill').cumcount()

# Is festival month
df['Is_Festival_Month']  = df['Month'].isin([10, 11]).astype(int)

# Is monsoon month
df['Is_Monsoon_Month']   = df['Month'].isin([6, 7, 8]).astype(int)

# Is summer month
df['Is_Summer_Month']    = df['Month'].isin([3, 4, 5]).astype(int)

print(f'Total features now: {df.shape[1]}')
print(df.columns.tolist())

Total features now: 32
['Date', 'Day', 'Opening_Stock', 'MS_Sold', 'HSD1_Sold', 'HSD2_Sold', 'HSD3_Sold', 'Total_Sold', 'Closing_Stock', 'Cash', 'Online', 'Card', 'Dip', 'Refill_Required', 'Year', 'Month', 'DayOfWeek', 'DayOfYear', 'WeekOfYear', 'Quarter', 'Is_Weekend', 'Day_Num', 'Target', 'Stock_Ratio', 'Rolling_7d_Sales', 'Rolling_3d_Sales', 'Prev_Closing', 'Prev_Total_Sold', 'Days_Since_Refill', 'Is_Festival_Month', 'Is_Monsoon_Month', 'Is_Summer_Month']


## Step 7 — Final summary & save

In [11]:
print('=== Final Dataset Summary ===')
print(f'Total rows         : {len(df)}')
print(f'Total columns      : {df.shape[1]}')
print(f'Date range         : {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Refill days (Yes)  : {df["Target"].sum()} ({df["Target"].mean()*100:.1f}%)')
print(f'No refill (No)     : {(df["Target"]==0).sum()} ({(df["Target"]==0).mean()*100:.1f}%)')
print(f'Avg Total_Sold     : {df["Total_Sold"].mean():.0f} L/day')
print(f'Min Closing_Stock  : {df["Closing_Stock"].min()} L')
print(f'Max Closing_Stock  : {df["Closing_Stock"].max()} L')

# Save clean data
df.to_csv('../data/clean_data.csv', index=False)
print('\n✅ Saved: data/clean_data.csv')

=== Final Dataset Summary ===
Total rows         : 806
Total columns      : 32
Date range         : 2024-01-01 to 2026-03-16
Refill days (Yes)  : 303 (37.6%)
No refill (No)     : 503 (62.4%)
Avg Total_Sold     : 4753 L/day
Min Closing_Stock  : 0 L
Max Closing_Stock  : 9165 L

✅ Saved: data/clean_data.csv
